# Preprocessing: Control Variables

This notebook preprocesses four control variable datasets into clean country-year format, ready to merge with the main dyadic panel.

**Control variables:**
1. GDP per capita (log) — World Bank
2. Population (log) — World Bank
3. Democracy (v2x_polyarchy) — V-Dem
4. Armed conflict (binary) — UCDP/PRIO

**Output:** `controls_merged.csv` keyed on `iso3` + `year`

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

# --- PATHS (adjust to your project structure) ---
PROJECT_ROOT = Path.cwd().parents[0]
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'controls'
OUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'controls'

YEAR_MIN = 1992
YEAR_MAX = 2013

print(Path.cwd())
print(PROJECT_ROOT)
print(RAW_DIR)
print(list(RAW_DIR.glob('*')))

/Users/mac/Documents/GitHub/Violent-Offenders-GPV---CSSci-/notebooks
/Users/mac/Documents/GitHub/Violent-Offenders-GPV---CSSci-
/Users/mac/Documents/GitHub/Violent-Offenders-GPV---CSSci-/data/raw/controls
[PosixPath('/Users/mac/Documents/GitHub/Violent-Offenders-GPV---CSSci-/data/raw/controls/worldbank-oda-bilateral-raw.csv'), PosixPath('/Users/mac/Documents/GitHub/Violent-Offenders-GPV---CSSci-/data/raw/controls/gdp-control-raw.csv'), PosixPath('/Users/mac/Documents/GitHub/Violent-Offenders-GPV---CSSci-/data/raw/controls/ucdp-control-raw.csv'), PosixPath('/Users/mac/Documents/GitHub/Violent-Offenders-GPV---CSSci-/data/raw/controls/v-dem-core-control-raw.csv'), PosixPath('/Users/mac/Documents/GitHub/Violent-Offenders-GPV---CSSci-/data/raw/controls/pop-worldbank-control-raw.csv')]


---
## 1. GDP per capita (World Bank)

In [6]:
# Load — World Bank CSVs have 4 header rows before the actual data
gdp_raw = pd.read_csv(RAW_DIR / 'gdp-control-raw.csv', skiprows=4)

# Reshape from wide (one column per year) to long
year_cols = [c for c in gdp_raw.columns if c.isdigit()]

gdp = gdp_raw.melt(
    id_vars=['Country Name', 'Country Code'],
    value_vars=year_cols,
    var_name='year',
    value_name='gdp_per_capita'
)
gdp['year'] = gdp['year'].astype(int)

# Filter to analysis window and drop missing
gdp = gdp[(gdp['year'] >= YEAR_MIN) & (gdp['year'] <= YEAR_MAX)]
gdp = gdp.dropna(subset=['gdp_per_capita'])

# Log-transform
gdp['gdp_per_capita_log'] = np.log(gdp['gdp_per_capita'])

# Rename
gdp = gdp.rename(columns={'Country Name': 'country_name', 'Country Code': 'iso3'})
gdp = gdp[['country_name', 'iso3', 'year', 'gdp_per_capita', 'gdp_per_capita_log']]

print(f'Shape: {gdp.shape}')
print(f'Countries: {gdp["iso3"].nunique()}')
print(f'Year range: {gdp["year"].min()} - {gdp["year"].max()}')
print(f'Missing: {gdp["gdp_per_capita"].isna().sum()}')
gdp.head()

Shape: (5584, 5)
Countries: 262
Year range: 1992 - 2013
Missing: 0


,country_name,iso3,year,gdp_per_capita,gdp_per_capita_log
8512,Aruba,ABW,1992,13892.605143,9.539112
8513,Africa Eastern and Southern,AFE,1992,732.910915,6.597024
8515,Africa Western and Central,AFW,1992,567.153365,6.340630
8516,Angola,AGO,1992,668.706017,6.505345
8517,Albania,ALB,1992,200.852220,5.302569


---
## 2. Population (World Bank)

In [7]:
pop_raw = pd.read_csv(RAW_DIR / 'pop-worldbank-control-raw.csv', skiprows=4)

year_cols = [c for c in pop_raw.columns if c.isdigit()]

pop = pop_raw.melt(
    id_vars=['Country Name', 'Country Code'],
    value_vars=year_cols,
    var_name='year',
    value_name='population'
)
pop['year'] = pop['year'].astype(int)

pop = pop[(pop['year'] >= YEAR_MIN) & (pop['year'] <= YEAR_MAX)]
pop = pop.dropna(subset=['population'])

# Log-transform
pop['population_log'] = np.log(pop['population'])

pop = pop.rename(columns={'Country Name': 'country_name', 'Country Code': 'iso3'})
pop = pop[['country_name', 'iso3', 'year', 'population', 'population_log']]

print(f'Shape: {pop.shape}')
print(f'Countries: {pop["iso3"].nunique()}')
print(f'Year range: {pop["year"].min()} - {pop["year"].max()}')
print(f'Missing: {pop["population"].isna().sum()}')
pop.head()

Shape: (5830, 5)
Countries: 265
Year range: 1992 - 2013
Missing: 0


,country_name,iso3,year,population,population_log
8512,Aruba,ABW,1992,69005.0,11.141934
8513,Africa Eastern and Southern,AFE,1992,329082707.0,19.611820
8514,Afghanistan,AFG,1992,13278974.0,16.401692
8515,Africa Western and Central,AFW,1992,221191375.0,19.214539
8516,Angola,AGO,1992,12423712.0,16.335117


---
## 3. Democracy — V-Dem (Electoral Democracy Index)

In [9]:
# Load V-Dem Core CSV
# NOTE: update filename if yours differs (check your download folder)
vdem_raw = pd.read_csv(RAW_DIR / 'v-dem-core-control-raw.csv')

print(f'Raw shape: {vdem_raw.shape}')
print(f'\nColumns containing "country" or "year" or "polyarchy":')
print([c for c in vdem_raw.columns if any(k in c.lower() for k in ['country', 'year', 'polyarchy'])])

Raw shape: (27913, 1818)

Columns containing "country" or "year" or "polyarchy":
['country_name', 'country_text_id', 'country_id', 'year', 'v2x_polyarchy', 'v2x_polyarchy_codelow', 'v2x_polyarchy_codehigh', 'v2x_polyarchy_sd']


In [10]:
# Select only the columns we need
vdem = vdem_raw[['country_name', 'country_text_id', 'year', 'v2x_polyarchy']].copy()
vdem = vdem.rename(columns={'country_text_id': 'iso3'})

# Filter to analysis window
vdem = vdem[(vdem['year'] >= YEAR_MIN) & (vdem['year'] <= YEAR_MAX)]
vdem = vdem.dropna(subset=['v2x_polyarchy'])

# V-Dem uses some non-standard country codes — fix to ISO3
vdem_iso3_fixes = {
    'BUR': 'MMR',   # Burma -> Myanmar
    'HAI': 'HTI',   # Haiti
    'ROK': 'KOR',   # Republic of Korea
    'DRV': 'VNM',   # Vietnam
    'RVN': 'VNM',   # Vietnam
    'ZAN': 'TZA',   # Zanzibar -> Tanzania
    'PSG': 'PSE',   # Palestine/Gaza
    'SML': 'SOM',   # Somaliland -> Somalia
    'HSD': 'IND',   # Hyderabad -> India (historical)
}
vdem['iso3'] = vdem['iso3'].replace(vdem_iso3_fixes)

# If duplicates after fixing (e.g. Vietnam), take mean per country-year
vdem = vdem.groupby(['iso3', 'year'], as_index=False).agg({
    'country_name': 'first',
    'v2x_polyarchy': 'mean'
})

print(f'Shape: {vdem.shape}')
print(f'Countries: {vdem["iso3"].nunique()}')
print(f'Year range: {vdem["year"].min()} - {vdem["year"].max()}')
print(f'v2x_polyarchy range: {vdem["v2x_polyarchy"].min():.3f} - {vdem["v2x_polyarchy"].max():.3f}')
vdem.head()

Shape: (3861, 4)
Countries: 177
Year range: 1992 - 2013
v2x_polyarchy range: 0.014 - 0.923


,iso3,year,country_name,v2x_polyarchy
0,AFG,1992,Afghanistan,0.096
1,AFG,1993,Afghanistan,0.094
2,AFG,1994,Afghanistan,0.094
3,AFG,1995,Afghanistan,0.094
4,AFG,1996,Afghanistan,0.076


---
## 4. Armed Conflict (UCDP/PRIO)

In [11]:
ucdp_raw = pd.read_csv(RAW_DIR / 'ucdp-control-raw.csv')

print(f'Raw shape: {ucdp_raw.shape}')
print(f'Year range: {ucdp_raw["year"].min()} - {ucdp_raw["year"].max()}')
print(f'\nSample locations: {ucdp_raw["location"].head(5).tolist()}')
print(f'Sample gwno_loc: {ucdp_raw["gwno_loc"].head(5).tolist()}')

Raw shape: (2752, 28)
Year range: 1946 - 2024

Sample locations: ['India', 'India', 'Egypt, Israel', 'Egypt, Israel', 'Egypt, Israel']
Sample gwno_loc: ['750', '750', '651, 666', '651, 666', '651, 666']


In [12]:
# Gleditsch-Ward code -> ISO3 mapping
gw_to_iso3 = {
    '2': 'USA', '20': 'CAN', '40': 'CUB', '41': 'HTI', '42': 'DOM',
    '51': 'JAM', '52': 'TTO', '55': 'GRD', '70': 'MEX', '90': 'GTM',
    '91': 'HND', '92': 'SLV', '93': 'NIC', '94': 'CRI', '95': 'PAN',
    '100': 'COL', '101': 'VEN', '110': 'GUY', '115': 'SUR',
    '130': 'ECU', '135': 'PER', '140': 'BRA', '145': 'BOL',
    '150': 'PRY', '155': 'CHL', '160': 'ARG', '165': 'URY',
    '200': 'GBR', '205': 'IRL', '210': 'NLD', '211': 'BEL', '212': 'LUX',
    '220': 'FRA', '225': 'CHE', '230': 'ESP', '235': 'PRT',
    '255': 'DEU', '260': 'DEU', '265': 'DEU',
    '290': 'POL', '305': 'AUT', '310': 'HUN',
    '315': 'CZE', '316': 'CZE', '317': 'SVK',
    '325': 'ITA', '338': 'MLT', '339': 'ALB',
    '341': 'MNE', '343': 'MKD', '344': 'HRV', '345': 'SRB',
    '346': 'BIH', '349': 'SVN', '350': 'GRC', '352': 'CYP',
    '355': 'BGR', '359': 'MDA', '360': 'ROU',
    '365': 'RUS', '366': 'EST', '367': 'LVA', '368': 'LTU',
    '369': 'UKR', '370': 'BLR', '371': 'ARM', '372': 'GEO', '373': 'AZE',
    '375': 'FIN', '380': 'SWE', '385': 'NOR', '390': 'DNK', '395': 'ISL',
    '402': 'CPV', '404': 'GNB', '411': 'GIN',
    '420': 'GMB', '432': 'MLI', '433': 'SEN', '434': 'BEN',
    '435': 'MRT', '436': 'NER', '437': 'CIV', '438': 'GIN',
    '439': 'BFA', '450': 'LBR', '451': 'SLE', '452': 'GHA',
    '461': 'TGO', '471': 'CMR', '475': 'NGA',
    '481': 'GAB', '482': 'CAF', '483': 'TCD', '484': 'COG', '490': 'COD',
    '500': 'UGA', '501': 'KEN', '510': 'TZA',
    '516': 'BDI', '517': 'RWA', '520': 'SOM', '522': 'DJI',
    '530': 'ETH', '531': 'ERI', '540': 'AGO', '541': 'MOZ',
    '551': 'ZMB', '552': 'ZWE', '553': 'MWI',
    '560': 'ZAF', '565': 'NAM', '570': 'LSO', '571': 'BWA',
    '572': 'SWZ', '580': 'MDG', '581': 'COM', '590': 'MUS',
    '600': 'MAR', '615': 'DZA', '616': 'TUN', '620': 'LBY',
    '625': 'SDN', '626': 'SSD', '630': 'IRN', '640': 'TUR',
    '645': 'IRQ', '651': 'EGY', '652': 'SYR', '660': 'LBN',
    '663': 'JOR', '666': 'ISR', '670': 'SAU',
    '678': 'YEM', '679': 'YEM', '680': 'YEM',
    '690': 'KWT', '692': 'BHR', '694': 'QAT', '696': 'ARE', '698': 'OMN',
    '700': 'AFG', '701': 'TKM', '702': 'TJK', '703': 'KGZ',
    '704': 'UZB', '705': 'KAZ', '710': 'CHN', '712': 'MNG',
    '713': 'TWN', '730': 'KOR', '731': 'PRK', '732': 'KOR',
    '740': 'JPN', '750': 'IND', '751': 'IND',
    '760': 'BTN', '770': 'PAK', '771': 'BGD',
    '775': 'MMR', '780': 'LKA', '781': 'MDV', '790': 'NPL',
    '800': 'THA', '811': 'KHM', '812': 'LAO',
    '816': 'VNM', '817': 'VNM',
    '820': 'MYS', '830': 'SGP', '835': 'BRN',
    '840': 'PHL', '850': 'IDN', '860': 'TLS',
    '900': 'AUS', '910': 'PNG', '920': 'NZL',
    '935': 'VUT', '940': 'SLB', '946': 'FJI'
}

# Filter to analysis window
ucdp = ucdp_raw[(ucdp_raw['year'] >= YEAR_MIN) & (ucdp_raw['year'] <= YEAR_MAX)].copy()

# Explode multi-country conflicts (gwno_loc can be "651, 666")
rows = []
for _, row in ucdp.iterrows():
    for gw in str(row['gwno_loc']).split(', '):
        gw = gw.strip()
        if gw in gw_to_iso3:
            rows.append({
                'iso3': gw_to_iso3[gw],
                'year': row['year'],
                'intensity_level': row['intensity_level']
            })

conflict_expanded = pd.DataFrame(rows)

# Aggregate to country-year: binary flag + max intensity
conflict = conflict_expanded.groupby(['iso3', 'year']).agg(
    armed_conflict=('intensity_level', 'count'),
    conflict_intensity=('intensity_level', 'max')
).reset_index()

conflict['armed_conflict'] = (conflict['armed_conflict'] > 0).astype(int)

print(f'Shape: {conflict.shape}')
print(f'Countries with conflict: {conflict["iso3"].nunique()}')
print(f'Year range: {conflict["year"].min()} - {conflict["year"].max()}')
print(f'\nTop 10 conflict countries (total conflict-years):')
print(conflict.groupby('iso3')['armed_conflict'].sum().sort_values(ascending=False).head(10))
conflict.head()

Shape: (636, 4)
Countries with conflict: 72
Year range: 1992 - 2013

Top 10 conflict countries (total conflict-years):
iso3
AFG    22
DZA    22
TUR    22
SDN    22
PHL    22
MMR    22
IND    22
COL    22
ISR    21
ETH    21
Name: armed_conflict, dtype: int64


,iso3,year,armed_conflict,conflict_intensity
0,AFG,1992,1,2
1,AFG,1993,1,2
2,AFG,1994,1,2
3,AFG,1995,1,2
4,AFG,1996,1,2


---
## 5. Merge all controls

In [13]:
# Start with GDP
controls = gdp[['iso3', 'year', 'gdp_per_capita', 'gdp_per_capita_log']].copy()

# Add population
controls = controls.merge(
    pop[['iso3', 'year', 'population', 'population_log']],
    on=['iso3', 'year'], how='outer'
)

# Add V-Dem democracy
controls = controls.merge(
    vdem[['iso3', 'year', 'v2x_polyarchy']],
    on=['iso3', 'year'], how='outer'
)

# Add conflict (left join — countries NOT in UCDP = no conflict = 0)
controls = controls.merge(
    conflict[['iso3', 'year', 'armed_conflict', 'conflict_intensity']],
    on=['iso3', 'year'], how='left'
)
controls['armed_conflict'] = controls['armed_conflict'].fillna(0).astype(int)
controls['conflict_intensity'] = controls['conflict_intensity'].fillna(0).astype(int)

# Final filter
controls = controls[(controls['year'] >= YEAR_MIN) & (controls['year'] <= YEAR_MAX)]

print(f'Final shape: {controls.shape}')
print(f'Countries: {controls["iso3"].nunique()}')
print(f'Year range: {controls["year"].min()} - {controls["year"].max()}')
print(f'\nMissingness:')
print(controls.isnull().sum())
print(f'\n% missing:')
print((controls.isnull().sum() / len(controls) * 100).round(1))
controls.head(10)

Final shape: (5874, 9)
Countries: 267
Year range: 1992 - 2013

Missingness:
iso3                     0
year                     0
gdp_per_capita         290
gdp_per_capita_log     290
population              44
population_log          44
v2x_polyarchy         2013
armed_conflict           0
conflict_intensity       0
dtype: int64

% missing:
iso3                   0.0
year                   0.0
gdp_per_capita         4.9
gdp_per_capita_log     4.9
population             0.7
population_log         0.7
v2x_polyarchy         34.3
armed_conflict         0.0
conflict_intensity     0.0
dtype: float64


,iso3,year,gdp_per_capita,gdp_per_capita_log,population,population_log,v2x_polyarchy,armed_conflict,conflict_intensity
0,ABW,1992,13892.605143,9.539112,69005.0,11.141934,NaN,0,0
1,ABW,1993,14700.959808,9.595668,73685.0,11.207555,NaN,0,0
2,ABW,1994,16055.287787,9.683794,77595.0,11.259258,NaN,0,0
3,ABW,1995,16548.717387,9.714064,79805.0,11.287341,NaN,0,0
4,ABW,1996,16620.954556,9.718420,83021.0,11.326849,NaN,0,0
5,ABW,1997,17750.009564,9.784141,86301.0,11.365596,NaN,0,0
6,ABW,1998,18828.087059,9.843105,88451.0,11.390204,NaN,0,0
7,ABW,1999,19216.197235,9.863509,89659.0,11.403769,NaN,0,0
8,ABW,2000,20681.023027,9.936972,90588.0,11.414077,NaN,0,0
9,ABW,2001,20740.132583,9.939826,91439.0,11.423427,NaN,0,0


In [14]:
# Save
controls.to_csv(OUT_DIR / 'controls_merged.csv', index=False)
print(f'Saved: {OUT_DIR / "controls_merged.csv"}')
print(f'\nTo merge with main panel:')
print('  panel = panel.merge(controls, left_on=["recipient_iso3", "year"], right_on=["iso3", "year"], how="left")')

Saved: /Users/mac/Documents/GitHub/Violent-Offenders-GPV---CSSci-/data/processed/controls/controls_merged.csv

To merge with main panel:
  panel = panel.merge(controls, left_on=["recipient_iso3", "year"], right_on=["iso3", "year"], how="left")
